In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle as pk
from IPython.display import Image, display
import os
import sys
from sklearn import decomposition


In [ ]:
main_dir = os.getcwd()
if "notebooks" in main_dir:
    main_dir = main_dir[:-10]
elif main_dir == "/content":
    main_dir = main_dir + "/GPA-source_code"
os.chdir(main_dir)
print(main_dir)

contrast_colors = {
    0: '#C71585', #pink
    1: '#1f77b4',  # blue
    2: '#2ca02c',  # green
    3: '#ff7f0e',  # orange
    4: '#8c564b',  # brown
    5: '#d62728',  # red 
    6: '#9467bd'  # purple (to be used for index 8)
}

data_dir = main_dir + '/data/'
#result_dir = '%s/assets/Transport_genes/' % main_dir


In [ ]:

#loads weights, biases and parameters of trained velocity field
def load_W(filename):
    with open(filename, "rb") as fr:
        W, b, p = pk.load(fr)

    W = [tf.Variable(w,  dtype=tf.float32) for w in W]
    b = [tf.Variable(b_,  dtype=tf.float32) for b_ in b]

    return W, b, p

    
def v(x, t, W, b):   # neural newtork for time-dependent vectorfield
    num_layers = len(W)
    activation_ftn = tf.nn.tanh
        
    h = tf.concat([x, t*tf.ones([x.shape[0], 1], dtype=tf.float32)], axis=1)
    for l in range(0,num_layers-1):
        h = activation_ftn(tf.add(tf.matmul(h, W[l]), b[l]))
    out=tf.add(tf.matmul(h, W[-1]), b[-1])

    return out

#returns a list of positions over time
def time_integration(x0, T, dt):
    x = tf.constant(x0, dtype=tf.float32)
    xs = [x0]
    for i in range(int(T/dt)):
        vv = v(x, dt*i, W, b)
        x += dt * vv
        xs.append(x.numpy())
    return xs



In [ ]:
from scripts.util.downstream import gene_dynamics_whole_saveonly
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_with_violin_plot_sample1_EMT
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_with_violin_plot_sample_3_stem
from scripts.util.downstream import create_pdf_from_gene_images
from scripts.util.downstream import Compute_and_Plot_FoldChange_MeanDiff_PValues
from scripts.util.downstream import difference_of_means_emt
from scripts.util.downstream import difference_of_means_stem
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_single_trajectory_EMT
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_single_trajectory_mESC
from scripts.util.downstream import Compare_Distribution_Trajectories_Intermediate_EMT
from scripts.util.downstream import Compare_Distribution_Trajectories_Intermediate_mESC

## Import Preprocessed Data and Define Dimensionality Reduction

- Please ensure you have run the `Preprocess_datasets.ipynb` notebook for the dataset you intend to use for downstream analysis.
- The preprocessed data should be saved in the `data/` folder with filenames ending in `_preprocessed.pkl`.
- When loading the preprocessed data below, make sure to use the correct and compatible filename corresponding to your dataset.
- We also load gene names from the gene expression matrix saved in the appropriate subfolder under the `data/` directory for each dataset. This step is optional and only needed if you wish to include **all genes** in the expression matrix. Otherwise, you can manually define a subset of genes of interest by assigning them to `gene_names`. For example, a gene subset for mESC data might look like:  
  ```python
  gene_names = ['SOX2', 'ESRRB', 'UTF1', 'EPAS1', 'FOXQ1']
- Each of the following cells corresponds to a specific dataset. Only run the cell for the dataset you're working with to avoid overwriting previously loaded data.

In [ ]:
## Loads mESC data

data_name = "mESC"

#loads the original data matrices
preprocessed_filename = data_dir + data_name + "_preprocessed.pkl"
with open(preprocessed_filename, "rb") as fr:
    try:
        time_label, full_matrix, projected_matrix, pca = pk.load(fr)
    except:
        time_label, full_matrix = pk.load(fr)   

pca = decomposition.PCA(n_components=2, random_state=0)
pca.fit(full_matrix)

d_red = 2
dimension_reduction = True

cls = set(time_label)
mats = {}
for c in cls:
    mats[c] = full_matrix[time_label == c].astype(float)

## Read Gene expression matrix (mESC data)


# select genes from time series data
if os.path.exists(data_dir + '/mESC data/'+ "Data_0_gene_expression_matrix.txt"):
    df_reduced_emt = pd.read_table(data_dir + '/mESC data/' + 'Data_0_gene_expression_matrix.txt', sep="\t")
    print(df_reduced_emt)
    col_selection_emt = df_reduced_emt.columns[1:]
elif os.path.exists(data_dir + 'Gene_Expression_Matrix.txt'):
    col_selection_emt = np.loadtxt(data_dir + 'non_overlap_72_genes.txt', dtype=str)
    df_reduced_emt = df.filter(items=np.concatenate((['id'], col_selection_emt)))
    print(df_reduced_emt)
    print("%d EMT Hallmark genes are excluded!" % (len(col_selection_emt)-len(df_reduced_emt.columns[1:])))
    col_selection_emt = df_reduced_emt.columns[1:]
    df_reduced_emt.to_csv(data_dir + 'submatrix_72_genes_expression_matrix.txt', sep='\t', index=False)

  
print("Original space dimension = ", len(col_selection_emt))

# Get the list of gene names (excluding the first column which contains cell names)
gene_names = df_reduced_emt.columns[1:].tolist()

# Print the list of gene names
print(gene_names)  

In [ ]:
## Loads EMT data

data_name = "emt_72"

#loads the original data matrices
preprocessed_filename = data_dir + data_name + "_preprocessed.pkl"
with open(preprocessed_filename, "rb") as fr:
    try:
        time_label, full_matrix, projected_matrix, pca = pk.load(fr)
    except:
        time_label, full_matrix = pk.load(fr)   

pca = decomposition.PCA(n_components=8, random_state=0)
pca.fit(full_matrix)

d_red = 8
dimension_reduction = True

cls = set(time_label)
mats = {}
for c in cls:
    mats[c] = full_matrix[time_label == c].astype(float)

## Read Gene expression matrix (EMT data)


# select genes from time series data
if os.path.exists(data_dir + '/EMT data/'+ "submatrix_72_genes_expression_matrix.txt"):
    df_reduced_emt = pd.read_table(data_dir + '/EMT data/' + 'submatrix_72_genes_expression_matrix.txt', sep="\t")
    print(df_reduced_emt)
    col_selection_emt = df_reduced_emt.columns[1:]
elif os.path.exists(data_dir + 'Gene_Expression_Matrix.txt'):
    col_selection_emt = np.loadtxt(data_dir + 'non_overlap_72_genes.txt', dtype=str)
    df_reduced_emt = df.filter(items=np.concatenate((['id'], col_selection_emt)))
    print(df_reduced_emt)
    print("%d EMT Hallmark genes are excluded!" % (len(col_selection_emt)-len(df_reduced_emt.columns[1:])))
    col_selection_emt = df_reduced_emt.columns[1:]
    df_reduced_emt.to_csv(data_dir + 'submatrix_72_genes_expression_matrix.txt', sep='\t', index=False)

  
print("Original space dimension = ", len(col_selection_emt))

# Get the list of gene names (excluding the first column which contains cell names)
gene_names = df_reduced_emt.columns[1:].tolist()

# Print the list of gene names
print(gene_names)  

## Load Trajectory Inferred by PROFET

- Make sure the reconstructed cell trajectory (generated by PROFET after the force-matching step) is saved as a `.pickle` file.  
- The code below will load that `.pickle` file.  
- For example, in the case of mESC data, the filename might be:
```
 EMT_dim2-f_Lip=5e-2-t_size=50-network=64_64_64.pickle
 ```
- Be sure to replace this with the filename corresponding to your own dataset.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.


In [ ]:
#loads the parameters and generates X1_trpts (mESC data)

folder_name = 'EMT_dim2-f_Lip=5e-2-t_size=50-network=64_64_64' #'times_10_particles_200_3'
result_dir = '%s/assets/Transport_genes/' % main_dir

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created.")

print("saving results to: ", result_dir)

d_red = 2
filename = result_dir + folder_name + ".pickle"
W, b, p = load_W(filename)

print("loaded: " + filename)
dt = p['numerical_ts'][-1]/200

X1_trpts = time_integration(pca.transform(mats[0]), T = p['numerical_ts'][-1], dt = dt)
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]


In [ ]:
#loads the parameters and generates X1_trpts (EMT data)

folder_name = '72GS_dim8-f_Lip=5e-2-t_size=50-network=64_64_64' # EMT data pickle file name
result_dir = '%s/assets/Transport_genes/' % main_dir

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created.")

print("saving results to: ", result_dir)

d_red = 8
filename = result_dir + folder_name + ".pickle"
W, b, p = load_W(filename)

print("loaded: " + filename)
dt = p['numerical_ts'][-1]/200

X1_trpts = time_integration(pca.transform(mats[0]), T = p['numerical_ts'][-1], dt = dt)
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]


## Downstream Analysis of Average Gene Dynamics for Each Gene with Confidence Intervals

- This section plots the **mean gene expression dynamics** across all predicted cell trajectories and compares them with the observed means.
- A **95% confidence interval** is included around each mean trajectory to indicate variability across cells.
- `source_t` and `target_t` define the earliest and latest time points of the reconstructed trajectory, based on the `.pickle` file loaded in the previous section.
- `intermediate_t` specifies the intermediate time points present in the input data.
- You can customize the temporal window for plotting using `start_i` (start time) and `max_i` (end time).
- The temporal resolution of the trajectory is controlled by `index`, where a smaller value (e.g., `index = 1`) gives the highest resolution.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.


In [ ]:
## For mESC data
## Average gene dynamics

gene_of_interest = gene_names

source_t, target_t = 0, 4
intermediate_t = [1,2,3]


start_i = 0
max_i = 200
index = 1

exp_memo = folder_name



# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        img_src = f"{result_dir}{folder_name}_average_gene_expression_{gene}.png"
        Average_gene_dynamics_whole_saveonly(pca,gene_names,source_t, target_t,X1_trpts,mats, gene, index,p, max_i,
                              intermediate_t = intermediate_t, img_src = img_src)
    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")

In [ ]:
## For EMT data
## Average gene dynamics

gene_of_interest = gene_names

source_t, target_t = 0, 4
intermediate_t = [2]


start_i = 0
max_i = 200
index = 1

exp_memo = folder_name



# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        img_src = f"{result_dir}{folder_name}_average_gene_expression_{gene}.png"
        Average_gene_dynamics_whole_saveonly(pca,gene_names,source_t, target_t,X1_trpts,mats, gene, index,p, max_i,
                              intermediate_t = intermediate_t, img_src = img_src)
    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")

## Downstream Analysis: Average Gene Dynamics Within Each Subtrajectory

- This section plots the **mean gene expression dynamics** within each subtrajectory and compares them with the observed means.
- A **95% confidence interval** is included around each mean trajectory to indicate variability across cells.
- Subtrajectories are defined in the notebook *"Downstream analysis – Trajectory visualization and subtrajectory classification"*.
- Please ensure that you've run that notebook to generate the subtrajectory classification annotations, saved as:
  - `_X1_hat_clusters.csv` (classified by **fates**, for mESC data), or
  - `_X2_hat_clusters.csv` (classified by **ancestors**, for EMT data).
- It is recommended to load the cluster annotation file using the following format:
  ```python
  f"{result_dir}{exp_memo}_X1_hat_clusters.csv"
- `optimal_k` corresponds to the number of subtrajectory classes.

- `source_t` and `target_t` define the earliest and latest time points of the reconstructed trajectory, based on the `.pickle` file loaded in the previous section.
- `intermediate_t` specifies the intermediate time points present in the input data.
- You can customize the temporal window for plotting using `start_i` (start time) and `max_i` (end time).
- The temporal resolution of the trajectory is controlled by `index`, where a smaller value (e.g., `index = 1`) gives the highest resolution.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.

In [ ]:
## For mESC data
## Average of subtrajecotries 

exp_memo = folder_name
gene_of_interest = gene_names



source_t, target_t = 0, 4
intermediate_t = [1,2,3]


start_i = 0
max_i = 200
index = 1

cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_clusters.csv"
optimal_k = 2


# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        img_src = f"{result_dir}{folder_name}_subtrajectories_violin_plots_{gene}.png" 
        Average_gene_dynamics_whole_saveonly_with_violin_plot_sample_3_stem(pca, gene_names, source_t, target_t,X1_trpts,mats,optimal_k, gene, index, p, max_i, 
                                                                    intermediate_t=intermediate_t, img_src = img_src, cluster_save_path = cluster_save_path)
    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")




In [ ]:
## For EMT data
## Average of subtrajecotries 

exp_memo = folder_name
gene_of_interest = gene_names


source_t, target_t = 0, 4
intermediate_t = [2]


start_i = 0
max_i = 200
index = 1

cluster_save_path = f"{result_dir}{exp_memo}_X2_hat_clusters.csv"
optimal_k = 2


# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        img_src = f"{result_dir}{folder_name}_subtrajectories_violin_plots_{gene}.png" 
        Average_gene_dynamics_whole_saveonly_with_violin_plot_sample1_EMT(pca, gene_names, source_t, target_t,X1_trpts,mats, gene, index, p, max_i,
                              intermediate_t = intermediate_t, img_src = img_src, cluster_save_path = cluster_save_path)
    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")



## Dynamics of p-values, Fold Change, and Mean Differences Across Subtrajectories (with CSV Outputs and Visualizations)

- This section computes statistical metrics—**p-values**, **fold change**, and **mean differences**—across subtrajectories.
- All statistical results and visualizations are saved in the `subtraj` subfolder, which resides within the `result_dir`.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.


In [ ]:
## Compute statistics for mESC data

subtraj = os.path.join(result_dir, 'subtraj')
# Create necessary directories
os.makedirs(subtraj, exist_ok=True)

exp_memo = folder_name
cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_clusters.csv"
gene_of_interest = gene_names

start_i = 0
index = 1
max_i = 200


for gene in gene_of_interest:
    Compute_and_Plot_FoldChange_MeanDiff_PValues(
        pca, gene_names, X1_trpts, gene, index, p, max_i,
        subtraj_dir=subtraj,
        cluster_save_path=cluster_save_path
    )


difference_of_means_stem(gene_names, subtraj)

In [ ]:
## Compute statistics for EMT data

subtraj = os.path.join(result_dir, 'subtraj')
# Create necessary directories
os.makedirs(subtraj, exist_ok=True)

exp_memo = folder_name
cluster_save_path = f"{result_dir}{exp_memo}_X2_hat_clusters.csv"
gene_of_interest = gene_names

start_i = 0
index = 1
max_i = 200


for gene in gene_of_interest:
    Compute_and_Plot_FoldChange_MeanDiff_PValues(
        pca, gene_names, X1_trpts, gene, index, p, max_i,
        subtraj_dir=subtraj,
        cluster_save_path=cluster_save_path
    )


difference_of_means_emt(gene_names, subtraj)

## Plot Single-Gene Dynamics for Every Single Cell

- This section generates single-cell gene expression dynamics for each gene specified in `gene_of_interest`.
- In the example below, we use genes from `gene_names`, which contains all genes from the gene expression matrix.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.

In [ ]:
## For mESC data
## Gene expression dynamics for every single cell 

exp_memo = folder_name
gene_of_interest = gene_names

source_t, target_t = 0, 4
intermediate_t = [1,2,3]

start_i = 0
max_i = 200
index = 1

cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_clusters.csv"
optimal_k = 2


# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        subgroup_output_file = f"{result_dir}{exp_memo}_Individual_trajectories_violin_plot_{gene}.png"
        Average_gene_dynamics_whole_saveonly_single_trajectory_mESC(pca, gene_names, source_t, target_t,X1_trpts,mats,optimal_k, gene, index, p, max_i, intermediate_t = intermediate_t, subgroup_output_file = subgroup_output_file, cluster_save_path = cluster_save_path)

    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")



In [ ]:
## For EMT data
## Gene expression dynamics for every single cell 

exp_memo = folder_name
gene_of_interest = gene_names

source_t, target_t = 0, 4
intermediate_t = [2]


start_i = 0
max_i = 200
index = 1

cluster_save_path = f"{result_dir}{exp_memo}_X2_hat_clusters.csv"
optimal_k = 2

# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        subgroup_output_file = f"{result_dir}{exp_memo}_Individual_trajectories_violin_plot_{gene}.png"
        Average_gene_dynamics_whole_saveonly_single_trajectory_EMT(pca,gene_names,source_t, target_t,X1_trpts,mats, gene, index,p, max_i,
                              intermediate_t = intermediate_t, subgroup_output_file = subgroup_output_file, cluster_save_path = cluster_save_path)

    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")


## Distribution Comparison of Single Genes

- This section generates **density plots** to compare the predicted gene expression distributions from reconstructed trajectories with the observed data at the test time point.
- These plots help assess how well the predicted distributions align with the actual gene expression measurements.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.

In [ ]:
## For mESC data
## Distributions of gene expression at test time point 

exp_memo = folder_name
gene_of_interest = gene_names
source_t, target_t = 0, 4
start_i = 0
intermediate_t = [1,3]


# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        output_file = f"{result_dir}{exp_memo}KDE_Intermediate_Only_updated_{gene}.png"
        Compare_Distribution_Trajectories_Intermediate_mESC(pca,gene_names,source_t, target_t,X1_trpts,mats, gene ,p, intermediate_t = intermediate_t,  output_file = output_file)

    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")


In [ ]:
## For EMT data
## Distributions of gene expression at test time point 

exp_memo = folder_name
gene_of_interest = gene_names
source_t, target_t = 0, 4
start_i = 0
intermediate_t = [2]


# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        output_file = f"{result_dir}{exp_memo}KDE_Intermediate_Only_updated_{gene}.png"
        Compare_Distribution_Trajectories_Intermediate_EMT(pca,gene_names,source_t, target_t,X1_trpts,mats, gene ,p, intermediate_t = intermediate_t,  output_file = output_file)

    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")